# Step 5: Prioritize Targets
Combine all evidence and categorize drug targets.

In [ ]:
from gwas_explorer.prioritize import prioritize_targets
from gwas_explorer.config import PROCESSED_DIR, RESULTS_DIR
import pandas as pd

druggability = pd.read_parquet(PROCESSED_DIR / "t2d_druggability.parquet")
mr_results = pd.read_parquet(PROCESSED_DIR / "t2d_mr_results.parquet")
combined = druggability.merge(mr_results, on="GENE_SYMBOL", how="left")

prioritized = prioritize_targets(combined)
print("Target categories:")
print(prioritized["TARGET_CATEGORY"].value_counts())
prioritized.head(20)

In [ ]:
import matplotlib.pyplot as plt

cat_counts = prioritized["TARGET_CATEGORY"].value_counts()
colors = {
    "validated": "#2ecc71",
    "repurposing_candidate": "#3498db",
    "novel_druggable": "#f39c12",
    "novel_challenging": "#e74c3c",
}
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(cat_counts.index, cat_counts.values, color=[colors.get(c, "gray") for c in cat_counts.index])
ax.set_ylabel("Number of genes")
ax.set_title("T2D Drug Target Prioritization")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
prioritized.to_parquet(RESULTS_DIR / "t2d_target_prioritization.parquet", index=False)
csv_cols = [c for c in prioritized.columns if c != "DRUGS"]
prioritized[csv_cols].to_csv(RESULTS_DIR / "t2d_target_prioritization.csv", index=False)
print(f"Saved {len(prioritized)} prioritized targets")

In [ ]:
novel = prioritized[prioritized["TARGET_CATEGORY"] == "novel_druggable"]
print(f"\n=== Top Novel Druggable Targets ({len(novel)}) ===")
novel[["GENE_SYMBOL", "P_VALUE", "MR_SUPPORTS_CAUSALITY", "TRACTABILITY_SM"]].head(15)